# Generator

Builds the synthetic sales data every case runs on: daily actual sales for 30 series,
forecasts from two models, and a stable name for every row. The same seed always gives
the same files, byte for byte, and their fingerprint becomes the `snapshot_id`.

With the default seed these are exactly the numbers of the spreadsheet export used
to explore the data (checked when the generator was written).

In [ ]:
#| default_exp generator

In [ ]:
#| export
import csv
import hashlib
import io
import json
import random
from collections.abc import Sequence
from dataclasses import dataclass, fields
from pathlib import Path

from agentic_forecast_review.fixtures import Observation, seasonal_naive

## The recipe

10 stores × 3 categories. The category sets the typical units per day, a random store
size scales it, each weekday has its own multiplier (day 1 is a Monday), and each series
drifts slowly. Forecasts are issued on three days, each covering the next 28 days.

In [ ]:
#| export
SEED = 7
STORES = (
    "CA_1",
    "CA_2",
    "CA_3",
    "CA_4",
    "TX_1",
    "TX_2",
    "TX_3",
    "WI_1",
    "WI_2",
    "WI_3",
)
CATEGORY_LEVEL = {"FOODS": 60, "HOUSEHOLD": 25, "HOBBIES": 10}  # typical units per day
WEEKLY = (0.9, 0.85, 0.85, 0.9, 1.05, 1.25, 1.2)  # Mon..Sun multipliers
WEEKDAYS = ("Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun")
LAST_DAY = 196
ISSUE_DAYS = (112, 140, 168)  # forecasts are made on these days
HORIZON = 28  # each forecast covers the next 28 days
MODEL_VERSIONS = ("seasonal_naive_v1", "synthetic_v3")

## Rows and their names

One actual: a series' units on one day. One forecast: one model's units for a target
day, made on an issue day. Each row gets an ID a finding can cite, such as
`act__CA_2_FOODS__d0177` (actual, day 177) or
`fc__CA_2_FOODS__synthetic_v3__d0168__h09` (issued day 168, 9 days ahead).

In [ ]:
#| export
@dataclass(frozen=True)
class ActualRow:
    row_id: str
    series_id: str
    store: str
    category: str
    day: int
    value: int


@dataclass(frozen=True)
class ForecastRow:
    row_id: str
    series_id: str
    model_version: str
    issue_day: int
    target_day: int
    horizon: int
    value: int


def actual_row_id(series_id: str, day: int) -> str:
    return f"act__{series_id}__d{day:04d}"


def forecast_row_id(
    series_id: str, model_version: str, issue_day: int, horizon: int
) -> str:
    return f"fc__{series_id}__{model_version}__d{issue_day:04d}__h{horizon:02d}"

## Actual sales

Store level × category level × weekday × trend, plus noise; rounded to whole units and
never below 0.

In [ ]:
#| export
def build_actuals(rng: random.Random) -> tuple[ActualRow, ...]:
    """Daily units for every series, days 1 to LAST_DAY."""
    rows: list[ActualRow] = []
    for store in STORES:
        store_size = rng.uniform(0.7, 1.4)
        for category, base in CATEGORY_LEVEL.items():
            level = base * store_size
            trend = rng.uniform(-0.0005, 0.0015)  # fractional change per day
            series_id = f"{store}_{category}"
            for day in range(1, LAST_DAY + 1):
                mean = level * WEEKLY[(day - 1) % 7] * (1 + trend * day)
                value = max(0, round(mean + rng.gauss(0, 0.08 * level)))
                row_id = actual_row_id(series_id, day)
                rows.append(ActualRow(row_id, series_id, store, category, day, value))
    return tuple(rows)

## Forecasts

`seasonal_naive_v1` repeats the last week before the issue day. `synthetic_v3` is that
with a small upward drift and noise, so the two models differ a little. Series are
visited in alphabetical order; that order decides which noise each forecast gets, so it
must not change.

In [ ]:
#| export
def build_forecasts(
    actuals: Sequence[ActualRow], rng: random.Random
) -> tuple[ForecastRow, ...]:
    """Both models' forecasts, 1 to HORIZON days ahead of each issue day."""
    history: dict[str, list[Observation]] = {}
    for actual in actuals:
        observation = Observation(actual.series_id, actual.day, actual.value)
        history.setdefault(actual.series_id, []).append(observation)

    rows: list[ForecastRow] = []
    for series_id in sorted(history):
        for issue_day in ISSUE_DAYS:
            seen = history[series_id][:issue_day]
            for horizon in range(1, HORIZON + 1):
                naive = round(seasonal_naive(seen, period=7, horizon=horizon))
                drifted = naive * (1 + 0.002 * horizon) + rng.gauss(0, 1.5)
                values = {
                    "seasonal_naive_v1": naive,
                    "synthetic_v3": max(0, round(drifted)),
                }
                for model_version in MODEL_VERSIONS:
                    rows.append(
                        ForecastRow(
                            forecast_row_id(
                                series_id, model_version, issue_day, horizon
                            ),
                            series_id,
                            model_version,
                            issue_day,
                            issue_day + horizon,
                            horizon,
                            values[model_version],
                        )
                    )
    return tuple(rows)

## The snapshot

Actuals and forecasts from one seed, as three files: `actuals.csv`, `forecasts.csv`
and `snapshot.json` (what is inside, plus the fingerprint). The `snapshot_id` is
`snap_` and the first 12 characters of the SHA-256 of the two CSV files, so changing
any single number changes it.

In [ ]:
#| export
def _csv_text(
    row_type: type[ActualRow] | type[ForecastRow],
    rows: Sequence[ActualRow | ForecastRow],
) -> str:
    names = [field.name for field in fields(row_type)]
    buffer = io.StringIO()
    writer = csv.writer(buffer, lineterminator="\n")
    writer.writerow(names)
    writer.writerows([getattr(row, name) for name in names] for row in rows)
    return buffer.getvalue()


@dataclass(frozen=True)
class Snapshot:
    seed: int
    actuals: tuple[ActualRow, ...]
    forecasts: tuple[ForecastRow, ...]

    @property
    def snapshot_id(self) -> str:
        digest = hashlib.sha256()
        for text in (self._actuals_csv(), self._forecasts_csv()):
            digest.update(text.encode("utf-8") + b"\0")
        return "snap_" + digest.hexdigest()[:12]

    def meta(self) -> dict[str, object]:
        """What snapshot.json records."""
        return {
            "snapshot_id": self.snapshot_id,
            "seed": self.seed,
            "first_day": 1,
            "last_day": LAST_DAY,
            "issue_days": list(ISSUE_DAYS),
            "horizon": HORIZON,
            "model_versions": list(MODEL_VERSIONS),
            "series": sorted({actual.series_id for actual in self.actuals}),
            "n_actual_rows": len(self.actuals),
            "n_forecast_rows": len(self.forecasts),
        }

    def files(self) -> dict[str, str]:
        """The snapshot's three files, name → text."""
        return {
            "actuals.csv": self._actuals_csv(),
            "forecasts.csv": self._forecasts_csv(),
            "snapshot.json": json.dumps(self.meta(), indent=2) + "\n",
        }

    def _actuals_csv(self) -> str:
        return _csv_text(ActualRow, self.actuals)

    def _forecasts_csv(self) -> str:
        return _csv_text(ForecastRow, self.forecasts)

Build everything from one seed; write it to a folder.

In [ ]:
#| export
def generate(seed: int = SEED) -> Snapshot:
    """The whole snapshot. The same seed always gives the same snapshot."""
    rng = random.Random(seed)
    actuals = build_actuals(rng)
    return Snapshot(seed, actuals, build_forecasts(actuals, rng))


def write_snapshot(snapshot: Snapshot, folder: Path) -> None:
    """Write the three snapshot files into ``folder``."""
    folder.mkdir(parents=True, exist_ok=True)
    for name, text in snapshot.files().items():
        (folder / name).write_text(text, encoding="utf-8", newline="")

## Examples

What is in the snapshot: 30 series, 196 days each, and 2 models × 3 issue days × 28
days ahead for every series.

In [ ]:
snap = generate()
n_series = len({actual.series_id for actual in snap.actuals})
print(snap.snapshot_id)
print(
    n_series, "series,", len(snap.actuals), "actuals,", len(snap.forecasts), "forecasts"
)
assert (n_series, len(snap.actuals), len(snap.forecasts)) == (30, 5880, 5040)

Rows are found by their names.

In [ ]:
actuals_by_id = {actual.row_id: actual for actual in snap.actuals}
forecasts_by_id = {forecast.row_id: forecast for forecast in snap.forecasts}
print(actuals_by_id["act__CA_2_FOODS__d0177"])
print(forecasts_by_id["fc__CA_2_FOODS__synthetic_v3__d0168__h09"])
assert forecasts_by_id["fc__CA_2_FOODS__synthetic_v3__d0168__h09"].target_day == 177

Saturday is the busiest day, about 125% of an average day, as in the Excel analysis.

In [ ]:
overall = sum(actual.value for actual in snap.actuals) / len(snap.actuals)
share = {}
for index, weekday in enumerate(WEEKDAYS):
    units = [actual.value for actual in snap.actuals if (actual.day - 1) % 7 == index]
    share[weekday] = sum(units) / len(units) / overall
print({weekday: f"{value:.0%}" for weekday, value in share.items()})
assert max(share, key=share.__getitem__) == "Sat" and round(share["Sat"], 2) == 1.25

Save it as the development family's base snapshot. Running this again writes the same
bytes, so git shows no change.

In [ ]:
folder = Path("../cases/dev/fam_01")
write_snapshot(snap, folder)
print(sorted(path.name for path in folder.iterdir()))
print((folder / "snapshot.json").read_text()[:120], "...")